# StudentInsight AI — Exploratory Data Analysis (EDA)
### Student Performance & At-Risk Prediction
**ALGIONS AI LABS PRIVATE LIMITED — AI & Machine Learning Assessment**

---
## 1. Problem & Business Context
Educational institutions require early indicators to identify students who may encounter academic difficulties before upcoming major assessments.
- **Target**: `at_risk = 1` if the student scores `< 40.0%` on the next upcoming assessment; `0` otherwise.
- **Cutoff**: Only information strictly prior to the upcoming assessment is used (historical attendance, assignment completion, previous scores, trends).
- **Responsible Use**: Strictly an advisory decision-support tool for teachers, never for automated penalties or grade assignment.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.data_loader import load_data
from src.data_validation import validate_dataset

# Set visualization parameters
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.max_columns', 25)

# Load dataset
df = load_data('data/processed/student_prediction_data.csv')
print(f"Dataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


## 2. Dataset Dimensions, Types & Data Quality Validation
Let us check null values, data types, and run our comprehensive validation checks.


In [ ]:
# Data types & null values
print("--- Data Info ---")
print(df.info())

print("\n--- Data Quality Validation ---")
val_report = validate_dataset(df, strict=False)
print("Is Valid:", val_report["is_valid"])
print("Checks Summary:", {k: v["status"] for k, v in val_report["checks"].items()})


## 3. Summary Statistics of Academic Features

In [ ]:
df.describe().round(2)


## 4. Target Class Distribution (Academic At-Risk)
Analyzing the proportion of students scoring below 40% in upcoming assessments.


In [ ]:
target_dist = df['at_risk'].value_counts(normalize=True) * 100
print("Class Distribution (%):")
print(target_dist.round(2))

fig, ax = plt.subplots(figsize=(6, 4))
df['at_risk'].value_counts().plot(kind='bar', color=['#2563EB', '#DC2626'], ax=ax)
ax.set_title("Target Distribution: at_risk (0: Safe, 1: At-Risk)", fontweight='bold')
ax.set_xticklabels(["Not At-Risk (0)", "At-Risk (1)"], rotation=0)
ax.set_ylabel("Count")
plt.tight_layout()
plt.show()


## 5. Attendance & Assignment Completion vs At-Risk Status

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

sns.boxplot(x='at_risk', y='attendance_rate', data=df, palette=['#2563EB', '#DC2626'], ax=axes[0])
axes[0].set_title("Attendance Rate (%) by Risk Status", fontweight='bold')
axes[0].set_xticklabels(["Not At-Risk", "At-Risk"])

sns.boxplot(x='at_risk', y='assignment_completion_rate', data=df, palette=['#2563EB', '#DC2626'], ax=axes[1])
axes[1].set_title("Assignment Completion (%) by Risk Status", fontweight='bold')
axes[1].set_xticklabels(["Not At-Risk", "At-Risk"])

plt.tight_layout()
plt.show()


## 6. Correlation Analysis & Feature Interactions

In [ ]:
num_cols = ['attendance_rate', 'assignment_completion_rate', 'previous_score_1', 'previous_score_2', 'average_previous_score', 'next_assessment_score', 'at_risk']
corr = df[num_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', linewidths=0.5)
plt.title("Academic Feature Correlation Matrix", fontweight='bold')
plt.tight_layout()
plt.show()


## 7. Key Findings & Conclusion
1. **Class Imbalance**: Approximately 23.8% of students fall into the at-risk category (<40% next score).
2. **Attendance Signal**: Students in the at-risk group exhibit significantly lower mean attendance (<65%) compared to safe students (>85%).
3. **Formative Completion**: Incomplete assignments are heavily associated with upcoming failure risk.
4. **Leakage Prevention**: Next assessment score (`next_assessment_score`) is strongly correlated with previous scores and must be strictly quarantined from the predictive feature matrix.
